# Exercise 9.4 — For whom is the model worst?

*HDS MSc · Module 3 · the Python version of this exercise; the web page is at <https://inflamuob.github.io/HDS_M3/day9/day6-04-subgroups.html>.*

*Open it in Jupyter or VS Code and run it a cell at a time. Keep it in the folder that holds your `data` folder. Cells with `______` are blanks for you to fill in.*

> **A suggested addition, not part of the original module**
>
> This exercise has no page in the original version of the module. It asks a
> question Exercises 9.1 to 9.3 leave open: they explained the model as a whole
> and one patient at a time, but not whether it works equally well for
> different groups of patients. Accept, rewrite or remove it; nothing else
> depends on it.

A single test AUC describes the model for the 150 test patients as a whole.
The model can still serve some groups of patients much worse than others,
and the overall number will not show it. Here we split the test patients by
age and by BMI and compare.

> **The plots are for looking at**
>
> The code for every plot is included, so that you can run it and see the figure. You are not expected to write it yourself.

### Packages and data

In [ ]:
# Packages for this exercise. If one is missing, install it once from a terminal
# (in a Jupyter notebook, type %pip instead of pip):
# pip install pandas numpy matplotlib scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

The same data ([`data/diabetes_clean.csv`](data/diabetes_clean.csv)), split and random
forest as Exercises 9.1 to 9.3, with the predicted probability and the
predicted class (threshold 0.5) for each test patient.

In [ ]:
diabetes = pd.read_csv("data/diabetes_clean.csv")
diabetes = diabetes[["pregnant", "glucose", "mass", "pedigree", "age", "diabetes"]]
diabetes["diabetes"] = (diabetes["diabetes"] == "pos").astype(int)       # 1 = pos, 0 = neg

is_test = diabetes.index % 5 == 0               # every fifth person is held back
train = diabetes[~is_test]
test  = diabetes[is_test]

predictors = ["pregnant", "glucose", "mass", "pedigree", "age"]

rf = RandomForestClassifier(n_estimators=500, random_state=1)
rf = rf.fit(train[predictors], train["diabetes"])

test_preds = test.copy()
test_preds["prob"] = rf.predict_proba(test[predictors])[:, 1]   # probability of diabetes
test_preds["pred"] = rf.predict(test[predictors])               # predicted class, threshold 0.5

------------------------------------------------------------------------

### Part 1 — Performance by age group

Put the test patients into three age groups, and count them first:

In [ ]:
test_preds["age_group"] = pd.cut(test_preds["age"], bins=[20, 29, 39, np.inf],
                                 labels=["21-29", "30-39", "40 and over"])

test_preds.groupby("age_group", observed=False).agg(
    patients=("diabetes", "size"),
    with_diabetes=("diabetes", "sum"))

Now three measures in each group: the AUC (how well the model *ranks* the
patients of that group), and at the threshold of 0.5 the sensitivity (the
share of the people with diabetes it flags) and the specificity (the share of
the people without diabetes it clears). The loop takes one age group at a time.

In [ ]:
rows = []
for group, patients in test_preds.groupby("age_group", observed=False):
    has_diabetes = patients["diabetes"] == 1
    flagged      = patients["pred"] == 1
    rows.append({"age_group": group,
                 "roc_auc": roc_auc_score(patients["diabetes"], patients["prob"]),
                 "sens": (has_diabetes & flagged).sum() / has_diabetes.sum(),
                 "spec": (~has_diabetes & ~flagged).sum() / (~has_diabetes).sum()})

by_age = pd.DataFrame(rows)
by_age.round(3)

The model ranks the youngest patients best (AUC 0.818) but, at the threshold
of 0.5, flags only 7 of the 21 young people with diabetes (sensitivity
0.333), against 7 of 9 aged 30-39 and 16 of 23 aged 40 and over. The older
groups pay for their higher sensitivity with a lower specificity.

The plot shows why: the predicted probabilities for each age group, split by
whether the patient really has diabetes.

In [ ]:
groups = ["21-29", "30-39", "40 and over"]
fig, ax = plt.subplots(figsize=(7, 3.5))
for shift, outcome, colour in [(-0.2, 0, "tab:blue"), (0.2, 1, "tab:red")]:
    for i, group in enumerate(groups):
        values = test_preds.loc[(test_preds["age_group"] == group) &
                                (test_preds["diabetes"] == outcome), "prob"]
        ax.boxplot(values, positions=[i + shift], widths=0.3, showfliers=False)
        ax.scatter(np.full(len(values), i + shift), values, color=colour, alpha=0.5,
                   label=("diabetes" if outcome == 1 else "no diabetes") if i == 0 else None)
ax.axhline(0.5, color="black", linestyle="--")
ax.set_xticks(range(len(groups)))
ax.set_xticklabels(groups)
ax.set_xlabel("Age group")
ax.set_ylabel("Predicted probability of diabetes")
ax.legend(title="Diabetes", loc="center left", bbox_to_anchor=(1, 0.5))
fig.tight_layout()
plt.show()

**Question!** The youngest group has the best AUC and the lowest sensitivity.
How can both be true?

The AUC only asks whether, *within the group*, people with diabetes get higher
probabilities than people without. They do. But young patients' probabilities
are low across the board, because the model has learned that diabetes is less
common in the young, so most of the young people with diabetes still sit
below 0.5. A model can rank a group well and still miss most of its cases at a
fixed threshold.

> **Learning points**
>
> - One overall AUC can hide large differences between groups of patients.
> - Ranking within a group (AUC) and catching cases at a threshold (sensitivity) are different questions.
> - Always report how many patients, and how many with the outcome, each group's numbers rest on.

------------------------------------------------------------------------

### Part 2 — By BMI, and how far to trust a small group

**Your turn.** Do the same with two BMI groups: under 30, and 30 or over (the
usual cut-off for obesity).

In [ ]:
# Replace every ______ below, then run this cell.
test_preds["bmi_group"] = np.where(test_preds["mass"] < ______, "under 30", "30 and over")

rows = []
for group, patients in test_preds.groupby(______):
    has_diabetes = patients["diabetes"] == 1
    flagged      = patients["pred"] == 1
    rows.append({"bmi_group": group,
                 "patients": len(patients),
                 "with_diabetes": has_diabetes.sum(),
                 "roc_auc": roc_auc_score(patients["diabetes"], patients["prob"]),
                 "sens": (has_diabetes & flagged).sum() / has_diabetes.sum(),
                 "spec": (~has_diabetes & ~flagged).sum() / (~has_diabetes).sum()})

pd.DataFrame(rows).round(3)

**How far can we trust these numbers?** The BMI-under-30 group has only 9 patients with diabetes, and so has the
30-39 age group. A sensitivity of 0.333 there is 3 patients out of 9: one more
caught would make it 0.444, one fewer 0.222.

A rough rule: a sensitivity resting on $k$ patients with the outcome moves by
$1/k$ for each patient caught or missed, and its uncertainty is of the order of
$\sqrt{p(1-p)/k}$. With $k = 9$ and $p = 0.33$ that is about $\pm 0.16$ for one
standard error. Differences between small groups have to be large before they
mean anything, and the test set of a teaching dataset is too small to settle
questions of fairness; a real audit needs far more patients in each group.

**Question!** Suppose the youngest group's low sensitivity holds up in a large
data set. What could you do about it: change the model, change the threshold
for that group, or add a predictor? What would each of these cost?

> **Learning points**
>
> - Split the test set by clinically meaningful groups and report the numbers of patients, and of patients with the outcome, beside every metric.
> - With a handful of patients with the outcome, one patient more or less moves sensitivity a long way; treat differences between small groups as hypotheses, not findings.

## Stretch

> **Stretch — a threshold per group?**
>
> In the youngest group, find the threshold that gives a sensitivity of at
> least 0.7 on the test patients (try 0.2, 0.3, 0.4). How many more of the young
> people without diabetes are now flagged? Why would you need a separate
> validation set to choose that threshold honestly?